# Fluo-C3DL-MDA231 sequence 01: published U-Net workflow

This notebook records how to reproduce the published Ultrack workflow for `Fluo-C3DL-MDA231`. 

The published Cell Tracking Challenge submission used a 3D U-Net to predict cell foreground and boundary maps, then passed those maps to Ultrack.

The separate `03_mda231_raw_to_tracks.ipynb` notebook is an exploratory three-frame baseline based on image processing. Its results must not be described as a reproduction of the published U-Net benchmark.

## Reproduction source and limits

The authors published the original submission code, pretrained MDA231 checkpoint, dataset configuration, and sequence-specific run scripts in [`royerlab/ultrack_CTC_submission`](https://github.com/royerlab/ultrack_CTC_submission/tree/submission-2023-10). This notebook intentionally points to that implementation instead of recreating the neural network from copied code.

The official workflow requires an NVIDIA GPU with at least 12 GB of memory and was tested with 24 GB. The authors' Docker workflow also requires a Gurobi WLS license. Training a new model is not required because the pretrained `Fluo-C3DL-MDA231.ckpt` checkpoint is published.

The paper reports hidden-test results of CTB `0.683`, TRA `0.818`, and SEG `0.549`. Our public training data cannot directly reproduce those hidden-test scores.

## 1. Imports and software record

In [ ]:
import os
import shlex
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

print(f"Python: {sys.version.split()[0]}")
print(f"Ultrack: {version('ultrack')}")
print(f"Python executable: {sys.executable}")

## 2. Paths

Set these variables before starting Jupyter:

- `CTC_ROOT`: project data root containing the `ctc` directory;
- `ULTRACK_RUN_ROOT`: personal run-output directory; and
- `ULTRACK_CTC_REPO`: local clone of the authors' submission repository on branch `submission-2023-10`.

Model weights and generated data must remain outside this Git repository.

In [ ]:
required_variables = ["CTC_ROOT", "ULTRACK_RUN_ROOT", "ULTRACK_CTC_REPO"]
missing_variables = [name for name in required_variables if not os.environ.get(name)]
if missing_variables:
    raise RuntimeError(
        f"Set these variables and restart the kernel: {missing_variables}"
    )

CTC_ROOT = Path(os.environ["CTC_ROOT"]).expanduser()
ULTRACK_RUN_ROOT = Path(os.environ["ULTRACK_RUN_ROOT"]).expanduser()
ULTRACK_CTC_REPO = Path(os.environ["ULTRACK_CTC_REPO"]).expanduser()

DATASET = "Fluo-C3DL-MDA231"
SEQUENCE = "01"
EXPECTED_FRAME_COUNT = 12

DATA_ROOT = CTC_ROOT / "ctc"
RAW_DIR = DATA_ROOT / DATASET / SEQUENCE
RUN_ROOT = ULTRACK_RUN_ROOT / "mda231-seq01-official-unet"
STAGING_DATA_ROOT = RUN_ROOT / "data"
WORK_ROOT = RUN_ROOT / "work"
RESULT_DIR = STAGING_DATA_ROOT / DATASET / f"{SEQUENCE}_RES"
WEIGHTS_PATH = ULTRACK_CTC_REPO / "weights" / f"{DATASET}.ckpt"
RUN_SCRIPT = ULTRACK_CTC_REPO / f"{DATASET}-{SEQUENCE}.sh"

print(f"Raw input: {RAW_DIR}")
print(f"Official repository: {ULTRACK_CTC_REPO}")
print(f"Pretrained checkpoint: {WEIGHTS_PATH}")
print(f"Private run root: {RUN_ROOT}")
print(f"Expected result: {RESULT_DIR}")

## 3. Validate the raw sequence and official repository

In [ ]:
if not RAW_DIR.is_dir():
    raise FileNotFoundError(f"Raw sequence does not exist: {RAW_DIR}")
if not ULTRACK_CTC_REPO.is_dir():
    raise FileNotFoundError(
        f"Official submission repository does not exist: {ULTRACK_CTC_REPO}"
    )
if not RUN_SCRIPT.is_file():
    raise FileNotFoundError(f"Official run script does not exist: {RUN_SCRIPT}")

raw_paths = sorted(RAW_DIR.glob("t*.tif"))
if len(raw_paths) != EXPECTED_FRAME_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_FRAME_COUNT} raw volumes, found {len(raw_paths)}."
    )

first_volume = tifffile.imread(raw_paths[0])
print(f"Files: {raw_paths[0].name} to {raw_paths[-1].name}")
print(f"First volume shape (Z, Y, X): {first_volume.shape}")
print(f"First volume dtype: {first_volume.dtype}")
print(f"Checkpoint downloaded: {WEIGHTS_PATH.is_file()}")

## 4. Official preprocessing and inference

The sequence-specific script performs the following published workflow:

1. quantile normalization with lower quantile `0.0001` and upper quantile `0.9999`;
2. z-axis interpolation by a factor of `5`;
3. tiled 3D `EdgeUNet` inference using the pretrained MDA231 checkpoint;
4. foreground prediction and Gaussian-smoothed boundary prediction;
5. Ultrack segmentation, linking, and solving with the published MDA231 configuration; and
6. CTC export with scale `(0.2, 1, 1)`, margin `25`, and one dilation iteration.

The inference command uses tile shape `(150, 512, 512)`, no tile overlap, and boundary smoothing sigma `(0.5, 0.5, 0.5)`. Do not replace this stage with StarDist if the goal is comparison with the published workflow.

## 5. Prepare a private staging directory

The authors' script writes `<DATA_DIR>/Fluo-C3DL-MDA231/01_RES`. Do not point it directly at the shared read-only CTC directory. The commands below use a private staging directory and link only the raw sequence into it. Run them in a terminal after obtaining a GPU allocation and activating the official environment or container.

In [ ]:
staged_dataset_dir = STAGING_DATA_ROOT / DATASET
staged_raw_dir = staged_dataset_dir / SEQUENCE

commands = [
    f"mkdir -p {shlex.quote(str(staged_dataset_dir))}",
    f"mkdir -p {shlex.quote(str(WORK_ROOT))}",
    (f"ln -s {shlex.quote(str(RAW_DIR))} " f"{shlex.quote(str(staged_raw_dir))}"),
    f"cd {shlex.quote(str(ULTRACK_CTC_REPO))}",
    "bash download.sh",
    (
        f"DATA_DIR={shlex.quote(str(STAGING_DATA_ROOT))} "
        f"WK_DIR={shlex.quote(str(WORK_ROOT))} "
        f"bash {shlex.quote(RUN_SCRIPT.name)}"
    ),
]

print("Run these commands on an allocated GPU node:\n")
print("\n".join(commands))

### Before starting the run

Record the GPU model and available memory with `nvidia-smi`. Also record the submission repository commit, environment or container identifier, start/end times, and any deviations from the official script. The official repository pins an older Ultrack submission environment; running its scripts in the project's current Ultrack 0.8 environment is not automatically equivalent.

## 6. Inspect a completed result

Run this section only after the official workflow finishes. It validates the CTC files without loading the U-Net implementation into this notebook.

In [ ]:
mask_paths = sorted(RESULT_DIR.glob("mask*.tif"))
track_path = RESULT_DIR / "res_track.txt"

if not RESULT_DIR.is_dir():
    print(f"Result directory is not available yet: {RESULT_DIR}")
elif len(mask_paths) != EXPECTED_FRAME_COUNT:
    print(
        f"Incomplete result: expected {EXPECTED_FRAME_COUNT} masks, "
        f"found {len(mask_paths)}."
    )
elif not track_path.is_file():
    print(f"Tracking file is missing: {track_path}")
else:
    result_shapes = {tifffile.imread(path).shape for path in mask_paths}
    track_rows = [
        line
        for line in track_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print(f"Masks: {mask_paths[0].name} to {mask_paths[-1].name}")
    print(f"Mask shapes: {result_shapes}")
    print(f"Lineage rows: {len(track_rows)}")

In [ ]:
DISPLAY_FRAME = 0

result_path = RESULT_DIR / f"mask{DISPLAY_FRAME:03d}.tif"
raw_path = RAW_DIR / f"t{DISPLAY_FRAME:03d}.tif"

if not result_path.is_file():
    print(f"Result mask is not available yet: {result_path}")
else:
    raw_volume = tifffile.imread(raw_path)
    result_volume = tifffile.imread(result_path)

    if raw_volume.shape != result_volume.shape:
        raise ValueError(
            f"Raw and result shapes differ: "
            f"raw={raw_volume.shape}, result={result_volume.shape}."
        )

    z_index = raw_volume.shape[0] // 2
    raw_slice = raw_volume[z_index]
    result_slice = result_volume[z_index]
    masked_result = np.ma.masked_where(result_slice == 0, result_slice)

    figure, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(raw_slice, cmap="gray")
    axes[0].set_title("Raw image")
    axes[1].imshow(masked_result, cmap="nipy_spectral")
    axes[1].set_title("Ultrack result labels")
    axes[2].imshow(raw_slice, cmap="gray")
    axes[2].imshow(masked_result, cmap="nipy_spectral", alpha=0.5)
    axes[2].set_title("Ultrack labels over raw")

    for axis in axes:
        axis.axis("off")

    figure.suptitle(
        f"{DATASET} sequence {SEQUENCE}: frame {DISPLAY_FRAME:03d}, " f"z={z_index}",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()

## 7. Evaluation interpretation

A complete 12-frame public-training result may be evaluated against `01_GT` using the project's CTC evaluation script. Report those public-training metrics separately from the paper's hidden-test values. A three-frame subset must not be passed to the benchmark evaluator.

Until this official GPU workflow is completed, the project has:

- a completed or in-progress **raw-image 3D subset experiment** in `03_mda231_raw_to_tracks.ipynb`; and
- a documented **published U-Net reproduction path**, but no reproduced U-Net benchmark result.